# 專題核心模組：每日歷史新聞增量爬蟲與 FinBERT 情緒量化
本筆記本負責每日自動抓取五大權值股（台積電、鴻海、長榮、中鋼、中信金）之最新財經新聞，並透過預訓練之 NLP 模型進行多空情緒打分。
為防範前視偏誤（Look-Ahead Bias），本模組內建 13:30 收盤時間切分規則，確保盤後新聞之情緒遞延至次日生效。

In [5]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import os
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# 目標五檔個股
TARGET_STOCKS = {'2330': '台積電', '2317': '鴻海', '2603': '長榮', '2002': '中鋼', '2891': '中信金'}
CSV_FILE = 'sentiment_features_2026.csv'

print("📥 載入 FinBERT-Chinese 模型...")
tokenizer = BertTokenizer.from_pretrained('IDEA-CCNL/Erlangshen-Roberta-110M-Sentiment')
model = BertForSequenceClassification.from_pretrained('IDEA-CCNL/Erlangshen-Roberta-110M-Sentiment')

def get_finbert_score(text):
    if not text: return 0.0
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        score = probs[0][1].item() - probs[0][0].item()
    return round(score, 4)

def fetch_today_news():
    url = 'https://api.cnyes.com/media/api/v1/newslist/category/tw_stock'
    headers = {'User-Agent': 'Mozilla/5.0'}
    today = datetime.now()
    start_ts = int(datetime(today.year, today.month, today.day).timestamp())
    end_ts = int(today.timestamp())

    daily_news = []
    for page in range(1, 10):
        params = {'limit': 30, 'page': page, 'startAt': start_ts, 'endAt': end_ts}
        try:
            res = requests.get(url, headers=headers, params=params, timeout=10)
            if res.status_code != 200: break
            items = res.json().get('items', {}).get('data', [])
            if not items: break
            for item in items:
                daily_news.append({
                    'publish_time': datetime.fromtimestamp(item.get('publishAt')),
                    'title': str(item.get('title') or '').strip(),
                    'summary': str(item.get('summary') or '').strip(),
                })
            time.sleep(0.5)
        except Exception as e:
            print("爬蟲異常:", e)
            break
    return daily_news

def align_trading_date(publish_time):
    if publish_time.time() > datetime.strptime('13:30:00', '%H:%M:%S').time():
        return (publish_time + timedelta(days=1)).date()
    return publish_time.date()

def main():
    print(f"🚀 開始執行每日增量爬蟲: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    raw_news = fetch_today_news()
    filtered_data = []

    for news in raw_news:
        for code, name in TARGET_STOCKS.items():
            # 加上反向排除，避免把「李長榮」誤判成「長榮」
            if (name in news['title'] or code in news['title'] or name in news['summary']) and ('李長榮' not in news['title']):
                content = news['title'] + " " + news['summary']
                sentiment = get_finbert_score(content)
                aligned_date = align_trading_date(news['publish_time'])

                filtered_data.append({
                    'Date': aligned_date.strftime('%Y-%m-%d'),
                    'ticker': f"{code}.TW",
                    'Daily_Sentiment': sentiment
                })

    if not filtered_data:
        print("今天沒有這五檔股票的新聞。")
        return

    # 聚合成每日單一分數
    df_today = pd.DataFrame(filtered_data)
    df_today_agg = df_today.groupby(['ticker', 'Date'])['Daily_Sentiment'].mean().reset_index()

    # 讀取原本的歷史 CSV，把今天的新資料貼在最下面
    if os.path.exists(CSV_FILE):
        df_history = pd.read_csv(CSV_FILE)
        df_combined = pd.concat([df_history, df_today_agg], ignore_index=True)
        # 去除重複（如果同一天不小心跑了兩次，以最新的為主）
        df_combined = df_combined.drop_duplicates(subset=['ticker', 'Date'], keep='last')
    else:
        df_combined = df_today_agg

    df_combined.to_csv(CSV_FILE, index=False, encoding='utf-8-sig')
    print(f"✅ 成功將今日資料寫入 {CSV_FILE}！")

if __name__ == "__main__":
    main()

📥 載入 FinBERT-Chinese 模型...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🚀 開始執行每日增量爬蟲: 2026-10-12 20:25:34
✅ 成功將今日資料寫入 sentiment_features_2026.csv！
